In [4]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve
)

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv("burnout.csv")

# =========================
# TARGET VARIABLE
# =========================

y = (df["Attrition"] == "Yes").astype(int)

# =========================
# DROP USELESS COLUMNS
# =========================

drop_cols = [
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours"
]

X = df.drop(columns=["Attrition"] + drop_cols)

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# =========================
# CATEGORICAL + NUMERIC
# =========================

cat_cols = X_train.select_dtypes(include="object").columns.tolist()

num_cols = X_train.select_dtypes(exclude="object").columns.tolist()

# =========================
# PREPROCESSING
# =========================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# =========================
# MODEL
# =========================

model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="liblinear"
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# =========================
# TRAIN MODEL
# =========================

pipe.fit(X_train, y_train)

# =========================
# THRESHOLD TUNING
# =========================

val_proba = pipe.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_proba
)

best_threshold = 0.5
best_recall = -1

for p, r, t in zip(precision[:-1], recall[:-1], thresholds):

    if p >= 0.35 and r > best_recall:
        best_recall = r
        best_threshold = t

# =========================
# FINAL TEST EVALUATION
# =========================

test_proba = pipe.predict_proba(X_test)[:, 1]

test_pred = (test_proba >= best_threshold).astype(int)

print("\n===== RESULTS =====\n")

print("Threshold:", round(best_threshold, 4))

print("Recall:",
      round(recall_score(y_test, test_pred), 4))

print("Precision:",
      round(precision_score(y_test, test_pred), 4))

print("F1 Score:",
      round(f1_score(y_test, test_pred), 4))

print("Accuracy:",
      round(accuracy_score(y_test, test_pred), 4))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, test_pred))

# =========================
# RETRAIN ON FULL DATA
# =========================

pipe.fit(X, y)

# =========================
# SAVE MODEL
# =========================

artifact = {
    "pipeline": pipe,
    "threshold": float(best_threshold),
    "feature_columns": X.columns.tolist()
}

joblib.dump(
    artifact,
    "attrition_pipeline.joblib"
)

print("\nModel Saved Successfully!")

/tmp/ipykernel_13522/2061872917.py:68: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns.tolist()



===== RESULTS =====

Threshold: 0.4213
Recall: 0.75
Precision: 0.3293
F1 Score: 0.4576
Accuracy: 0.7104

Confusion Matrix:

[[130  55]
 [  9  27]]

Model Saved Successfully!
